In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm

%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
%%sh
# Download the data - you need to do this only once
wget --no-verbose --output-document=image.jpg https://github.com/chrirupp/cv_course/raw/main/data/image.jpg
wget --no-verbose --output-document=dog_cutout.png https://github.com/chrirupp/cv_course/raw/main/data/dog_cutout.png

# Lecture 12 - Video

Motion as a cue: frame differences and optical flow (classical and learned), what the first layer of a 3D CNN learns, and why video transformers factorise attention.

We use a **synthetic** clip that we render ourselves, so that we know the true motion of every pixel.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

In [ ]:
# A synthetic video with exact ground truth: a camera pans and zooms over a photo (the background),
# a dog walks through the scene and passes behind a pole that stands in the scene.
# Because we render it ourselves, we know where every pixel goes (flow) and when it is hidden (occlusion).
background = cv2.cvtColor(cv2.imread('image.jpg'), cv2.COLOR_BGR2RGB).astype(np.float32)
cutout = cv2.imread('dog_cutout.png', cv2.IMREAD_UNCHANGED)
cutout = cv2.resize(cutout, (0, 0), fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
dog_rgb = cv2.cvtColor(cutout[:, :, :3], cv2.COLOR_BGR2RGB).astype(np.float32)
dog_alpha = cutout[:, :, 3].astype(np.float32) / 255

H, W, T = 360, 480, 48
POLE_X, POLE_WIDTH = 640, 36            # the pole, in background (world) coordinates

def camera(t):
    """scale and offset: background point X appears at s * (X - o) in frame t"""
    return 0.5 * (1 + 0.003 * t), np.array([40 + 5.0 * t, 60 + 1.0 * t])

def dog_position(t):
    """top-left corner of the dog in frame t (image coordinates)"""
    return np.array([20 + 6.0 * t, 120 + 8 * np.sin(t / 5)])

def layers(t):
    """the masks of the dog and the pole in frame t (the pole is in front of the dog)"""
    s, o = camera(t)
    px, py = dog_position(t)
    shift = np.float32([[1, 0, px], [0, 1, py]])
    dog_mask = cv2.warpAffine(dog_alpha, shift, (W, H))
    pole_mask = np.zeros((H, W), np.float32)
    x0, x1 = s * (POLE_X - o[0]), s * (POLE_X + POLE_WIDTH - o[0])
    pole_mask[:, max(int(round(x0)), 0):max(int(round(x1)), 0)] = 1
    return dog_mask, pole_mask, shift

def render(t):
    s, o = camera(t)
    A = np.float32([[s, 0, -s * o[0]], [0, s, -s * o[1]]])
    frame = cv2.warpAffine(background, A, (W, H), flags=cv2.INTER_LINEAR)
    dog_mask, pole_mask, shift = layers(t)
    dog = cv2.warpAffine(dog_rgb, shift, (W, H))
    frame = frame * (1 - dog_mask[..., None]) + dog * dog_mask[..., None]
    frame = frame * (1 - pole_mask[..., None]) + np.array([70, 60, 55]) * pole_mask[..., None]
    return np.clip(frame, 0, 255).astype(np.uint8)

def ground_truth_tracks(points, t0=0):
    """positions (T, N, 2) and visibility (T, N) of points given in frame t0"""
    points = np.asarray(points, np.float32)
    dog_mask, pole_mask, _ = layers(t0)
    xi = np.clip(points[:, 0].round().astype(int), 0, W - 1)
    yi = np.clip(points[:, 1].round().astype(int), 0, H - 1)
    on_dog = dog_mask[yi, xi] > 0.5
    s0, o0 = camera(t0)
    world = points / s0 + o0                    # background points in world coordinates
    tracks, visible = np.zeros((T, len(points), 2), np.float32), np.zeros((T, len(points)), bool)
    for t in range(T):
        s, o = camera(t)
        tracks[t] = np.where(on_dog[:, None], points - dog_position(t0) + dog_position(t), s * (world - o))
        dog_mask, pole_mask, _ = layers(t)
        x, y = tracks[t, :, 0], tracks[t, :, 1]
        inside = (x >= 0) & (x <= W - 1) & (y >= 0) & (y <= H - 1)
        xi, yi = np.clip(x.round().astype(int), 0, W - 1), np.clip(y.round().astype(int), 0, H - 1)
        hidden = (pole_mask[yi, xi] > 0.5) | (~on_dog & (dog_mask[yi, xi] > 0.5))
        visible[t] = inside & ~hidden
    return tracks, visible

def ground_truth_flow(t):
    """dense optical flow from frame t to frame t+1 (H, W, 2)"""
    ys, xs = np.mgrid[0:H, 0:W].astype(np.float32)
    points = np.stack([xs.ravel(), ys.ravel()], axis=1)
    s0, o0 = camera(t)
    s1, o1 = camera(t + 1)
    flow = s1 * (points / s0 + o0 - o1) - points
    dog_mask, pole_mask, _ = layers(t)
    flow[dog_mask.ravel() > 0.5] = dog_position(t + 1) - dog_position(t)
    flow[pole_mask.ravel() > 0.5] = s1 * (points[pole_mask.ravel() > 0.5] / s0 + o0 - o1) - points[pole_mask.ravel() > 0.5]
    return flow.reshape(H, W, 2).astype(np.float32)

video = np.stack([render(t) for t in range(T)])
print('video:', video.shape)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def animate(frames, interval=80):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    im = ax.imshow(frames[0])
    ax.axis('off')
    anim = FuncAnimation(fig, lambda t: [im.set_data(frames[t])], frames=len(frames), interval=interval)
    plt.close()
    return HTML(anim.to_jshtml())

animate(video)

## Motion

The simplest motion cue is the difference between consecutive frames. It responds to moving edges, but it does not tell us *where* things move. Note that the camera moves too, so the whole background changes.

In [ ]:
ts = [10, 26]
fig, axs = plt.subplots(len(ts), 3, figsize=(18, 4.5 * len(ts)))
for row, t in zip(axs, ts):
    diff = np.abs(video[t + 1].astype(np.float32) - video[t].astype(np.float32)).mean(axis=2)
    row[0].imshow(video[t])
    row[0].set_title(f'Frame {t}')
    row[1].imshow(video[t + 1])
    row[1].set_title(f'Frame {t + 1}')
    row[2].imshow(diff, cmap='magma')
    row[2].set_title('|difference|')
    for ax in row:
        ax.axis('off')
plt.show()

## Optical flow

Optical flow assigns a 2D displacement to every pixel. We compare a classical method (Farnebäck, polynomial expansion + coarse-to-fine) with RAFT, a learned method that iteratively refines the flow using a 4D correlation volume. Flow is visualised with the usual colour wheel: the hue gives the direction, the saturation the magnitude.

The error measure is the *end-point error* (EPE): the mean Euclidean distance between estimated and true flow vectors.

In [ ]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
from torchvision.utils import flow_to_image
raft = raft_large(weights=Raft_Large_Weights.DEFAULT).to(device).eval()

def raft_flow(frame0, frame1):
    # RAFT expects images in [-1, 1] with sides divisible by 8
    x0 = torch.from_numpy(frame0).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(device)
    x1 = torch.from_numpy(frame1).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(device)
    with torch.no_grad():
        return raft(x0, x1)[-1][0].permute(1, 2, 0).cpu().numpy()   # the last of 12 refinement iterations

def farneback_flow(frame0, frame1):
    g0, g1 = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in (frame0, frame1)]
    return cv2.calcOpticalFlowFarneback(g0, g1, None, pyr_scale=0.5, levels=4, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

def flow_rgb(flow, max_flow=12):
    flow = torch.from_numpy(flow).permute(2, 0, 1).clamp(-max_flow, max_flow)
    flow[0, 0, 0], flow[1, 0, 1] = max_flow, max_flow       # fix the colour scale across images
    return flow_to_image(flow).permute(1, 2, 0).numpy()

def epe(flow, gt):
    return np.linalg.norm(flow - gt, axis=2).mean()

t = 26
gt = ground_truth_flow(t)
flows = {'Ground truth': gt, 'Farnebäck': farneback_flow(video[t], video[t + 1]), 'RAFT': raft_flow(video[t], video[t + 1])}
fig, axs = plt.subplots(1, 4, figsize=(24, 4.5))
axs[0].imshow(video[t])
axs[0].set_title(f'Frame {t}')
for ax, (name, flow) in zip(axs[1:], flows.items()):
    ax.imshow(flow_rgb(flow))
    ax.set_title(name if name == 'Ground truth' else f'{name}: EPE {epe(flow, gt):.2f} px')
for ax in axs:
    ax.axis('off')
savefig('video_optical_flow')
plt.show()

In [ ]:
# EPE over the whole clip
errors = {'Farnebäck': [], 'RAFT': []}
for t in tqdm(range(T - 1)):
    gt = ground_truth_flow(t)
    errors['Farnebäck'].append(epe(farneback_flow(video[t], video[t + 1]), gt))
    errors['RAFT'].append(epe(raft_flow(video[t], video[t + 1]), gt))
plt.figure(figsize=(10, 4))
for name, e in errors.items():
    plt.plot(e, label=f'{name} (mean {np.mean(e):.2f} px)')
plt.xlabel('frame')
plt.ylabel('end-point error (px)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## What does a 3D CNN learn?

The first layer of a 3D ResNet-18 trained on Kinetics-400 has 64 filters of size $3 \times 7 \times 7$ (time x height x width). Many filters are the same in all three time steps (appearance), others change over time: they respond to motion in a particular direction.

In [ ]:
from torchvision.models.video import r3d_18, R3D_18_Weights
r3d = r3d_18(weights=R3D_18_Weights.DEFAULT)
w = r3d.stem[0].weight.detach()          # (64, 3 colour channels, 3 time steps, 7, 7)
print('first layer:', tuple(w.shape))

# show every filter as three 7x7 frames side by side (one colour normalisation per filter)
# filters that look the same in all three frames respond to appearance; filters that change respond to motion or flicker
temporal_change = (w[:, :, 1:] - w[:, :, :-1]).flatten(1).norm(dim=1) / w.flatten(1).norm(dim=1)
order = temporal_change.argsort()
fig, axs = plt.subplots(8, 8, figsize=(16, 8))
for ax, idx in zip(axs.ravel(), order):
    f = w[idx].permute(1, 2, 3, 0).numpy()          # (time, 7, 7, colour)
    f = (f - f.min()) / (f.max() - f.min())
    strip = np.concatenate([f[0], np.ones((7, 1, 3)), f[1], np.ones((7, 1, 3)), f[2]], axis=1)
    ax.imshow(strip, interpolation='nearest')
    ax.axis('off')
fig.suptitle('First layer of a 3D ResNet-18 (Kinetics-400): 64 filters, each shown as t=0 | t=1 | t=2,\nsorted from static (top left) to changing over time (bottom right)')
savefig('video_3d_filters')
plt.show()

## The cost of attention over space and time

A Video Vision Transformer (ViViT) turns a clip into tokens by cutting it into *tubelets* (here $2 \times 16 \times 16$ pixels). Joint space-time attention compares every token with every other token, which grows quadratically with the number of frames. Factorised attention first attends within each frame (spatial), then across frames at each location (temporal).

In [ ]:
def attention_pairs(frames, size=224, patch=16, tubelet=2):
    n_t, n_s = frames // tubelet, (size // patch) ** 2
    joint = (n_t * n_s) ** 2
    factorised = n_t * n_s ** 2 + n_s * n_t ** 2
    return n_t * n_s, joint, factorised

frames = np.array([8, 16, 32, 64, 128, 256])
results = np.array([attention_pairs(f) for f in frames])
for f, (tokens, joint, fact) in zip(frames, results):
    print(f'{f:4d} frames: {tokens:6d} tokens, joint {joint/1e6:9.1f} M pairs, factorised {fact/1e6:7.1f} M pairs ({joint/fact:5.1f}x fewer)')

plt.figure(figsize=(7, 4))
plt.loglog(frames, results[:, 1], 'o-', label='joint space-time attention')
plt.loglog(frames, results[:, 2], 'o-', label='factorised (spatial, then temporal)')
plt.xlabel('# frames')
plt.ylabel('# attention pairs per layer')
plt.legend()
plt.grid(alpha=0.3, which='both')
savefig('video_attention_cost')
plt.show()